# 06 - Curate a golden dataset with evaluators and an LLM judge

Goal: score production traces with a code evaluator and an LLM-as-judge, rank them, and add the best to a golden dataset on a rolling basis.

A golden dataset is the set of examples you trust enough to test every release against. Hand-picking does not scale. Instead:

1. A cheap **code evaluator** scores every trace.
2. An **LLM judge** reads the best candidates and grades the answer itself, which code rules cannot do.
3. The top scorers join the dataset each cycle (for example monthly), looking back over a window. Nothing is added twice.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

In [ ]:
print(f"{ref.ensure_traffic(PROJECT)} root runs available")
dataset = lab_curate.ensure_dataset(client)
print("dataset:", dataset.name)

## Step 1: Write the code evaluator

`golden_score(root, trace_runs)` receives the root run of a trace and every run in that trace, and returns a score between 0 and 1:

- `0.0` if the root run errored, or its `policy_decision` metadata (from challenge 4) is anything other than `allow`.
- Otherwise start at `0.5`.
- Add `0.3 x` the mean retrieval score. `lab_curate.retrieval_scores(trace_runs)` gives every score in the trace; skip if there are none.
- Add `0.2` if no tool run, and no `.audit` span, in the trace has an error.

In [ ]:
# Reference solution: try writing this yourself first.
def golden_score(root, trace_runs):
    if root.error:
        return 0.0
    if enrich.meta_of(root).get("policy_decision", "allow") != "allow":
        return 0.0
    score = 0.5
    scores = lab_curate.retrieval_scores(trace_runs)
    if scores:
        score += 0.3 * sum(scores) / len(scores)
    if not any(r.error for r in trace_runs if r.run_type == "tool" or (r.name or "").endswith(".audit")):
        score += 0.2
    return round(score, 3)

## Step 2: Write the LLM judge

Code can check that a retrieval score is high. It cannot tell whether the answer is correct, grounded and well put. A judge model can.

`lab_curate.JUDGE_PROMPT` is a rubric prompt (printed below). Fill in `llm_judge(run, trace_runs)`: build the prompt from the question (`enrich.question_of`), the answer (`enrich.answer_of`) and the retrieved text,
call `lab_curate.ask_judge(prompt)` (Claude on Amazon Bedrock, temperature 0), and return `lab_curate.parse_judge(reply)`, which gives `(score 1-5, reason)`. If the model call fails, return `(0.0, <message>)`.

In [ ]:
print(lab_curate.JUDGE_PROMPT)

In [ ]:
# Reference solution: try writing this yourself first.
def llm_judge(run, trace_runs):
    context = "\n".join(d.get("page_content", "") for r in trace_runs if r.run_type == "retriever" or (r.name or "").startswith("search_kb")
                        for d in enrich.documents_of(r)) or "(none)"
    prompt = lab_curate.JUDGE_PROMPT.format(question=enrich.question_of(run), context=context,
                                            answer=enrich.answer_of(run))
    try:
        return lab_curate.parse_judge(lab_curate.ask_judge(prompt))
    except Exception as exc:
        return 0.0, str(exc)

Try it on one trace before running it over many:

In [ ]:
agent_traces = {r.trace_id for r in enrich.agent_runs(enrich.project_runs(client, PROJECT))}
sample = next(r for r in enrich.project_runs(client, PROJECT) if r.name == "bank_support_turn" and not r.error and r.trace_id in agent_traces)
sample_runs = [r for r in enrich.project_runs(client, PROJECT) if r.trace_id == sample.trace_id]
print("question:", enrich.question_of(sample))
print("answer:  ", enrich.answer_of(sample))
print("judge:   ", llm_judge(sample, sample_runs))

## Step 3: Score, judge, rank, and add (the rolling job)

`curate` is the whole monthly job:

1. Score every root run in the window with `golden_score` (cheap).
2. Judge only the best `judge_top` of those (costly).
3. Combine: `final = 0.4 x code score + 0.6 x judge score / 5`. Rank by `final`.
4. Add the top `top_k` at or above `min_final`. Each example records `source_run_id`, `golden_score`, `judge_score`, `judge_reason` and `curated_on` in its metadata, so a later cycle skips traces already in the dataset.

Scores are also written to LangSmith as `golden_score` and `judge_score` feedback, so you can sort on them. Feedback ids come from the run id, so a repeat run updates instead of duplicating.
Run it from a scheduler (cron, CI, a scheduled agent) with `window_days=30`.

In [ ]:
import uuid
from collections import defaultdict

def feedback(run, key, score, comment):
    client.create_feedback(run.id, key=key, score=score, comment=comment, session_id=run.session_id,
                           feedback_id=uuid.uuid5(uuid.NAMESPACE_URL, f"{run.id}:{key}"))

def curate(window_days=30, judge_top=10, top_k=5, min_final=0.7):
    runs = enrich.project_runs(client, PROJECT, hours=24 * window_days, limit=2000)
    agent_traces = {r.trace_id for r in enrich.agent_runs(runs)}
    runs = [r for r in runs if r.trace_id in agent_traces]
    by_trace = defaultdict(list)
    for r in runs:
        by_trace[r.trace_id].append(r)

    scored = [(golden_score(r, by_trace[r.trace_id]), r) for r in runs if r.name == "bank_support_turn"]
    for s, r in scored:
        feedback(r, "golden_score", s, "code evaluator: errors, policy, retrieval, tools")

    shortlist = sorted(scored, key=lambda pair: pair[0], reverse=True)[:judge_top]
    finals, extra = [], {}
    for code_score, r in shortlist:
        j, reason = llm_judge(r, by_trace[r.trace_id])
        feedback(r, "judge_score", j / 5, reason)
        final = round(0.4 * code_score + 0.6 * j / 5, 3)
        finals.append((final, r))
        extra[str(r.id)] = {"golden_score": code_score, "judge_score": j, "judge_reason": reason, "final_score": final}

    best = [pair for pair in sorted(finals, key=lambda pair: pair[0], reverse=True)[:top_k] if pair[0] >= min_final]
    added = lab_curate.add_runs(client, dataset, [r for _, r in best], enrich.answer_of, "evaluator", extra)
    print(f"scored {len(scored)}, judged {len(shortlist)}, selected {len(best)}, added {added} new examples to {dataset.name}")
    for final, r in best:
        print(f"  {final:.2f}  {extra[str(r.id)]['judge_score']:.0f}/5  {extra[str(r.id)]['judge_reason'][:80]}")
    return best

best = curate()

Run it again. The same traces are selected, and nothing new is added:

In [ ]:
_ = curate()

## Step 4: See it in LangSmith

Open the dataset and read an example's metadata: the judge's reason is there. In the project, sort traces by the `judge_score` feedback column.

LangSmith can also do the selection server-side with an automation rule that adds runs matching a filter (for example `judge_score >= 0.8`) to a dataset. The code above is the explicit version,
which makes the ranking and the rolling window easy to see and change.

In [ ]:
lab_links.dataset_link(lab_curate.DATASET_NAME, hint="open an example: source_run_id, golden_score, judge_score and judge_reason are in its metadata")
lab_links.project_link(hint="sort the trace list by the judge_score feedback column")

**Result:** a dataset with up to five judged examples, and a second `curate()` run that adds none.

Next: challenge 7 lets human reviewers add to the same dataset, and validates coverage.